In [1]:
# import the required libraries
import sys
from pathlib import Path

import ipywidgets as widgets
from czitools.metadata_tools import czi_metadata as czimd
from czitools.utils import misc, planetable
from ipyfilechooser import FileChooser
from IPython.display import HTML, display

# Make the shared utility importable from any workshop subdirectory.
for utility_dir in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (utility_dir / "workshop_utils.py").is_file():
        if str(utility_dir) not in sys.path:
            sys.path.insert(0, str(utility_dir))
        break
else:
    raise FileNotFoundError("Start from the workshop root or a subdirectory.")

from workshop_utils import initialize_workshop

workshop_root, project_dir, data_dir, model_dir = initialize_workshop(
    "czi_handling"
)

Kernel Python: f:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\.pixi\envs\default\python.exe
Workshop: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11
Project: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\examples\czi_handling
Data: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\data
Model: F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\data\DL_models


In [2]:
# choose local file
fc = FileChooser()
fc.default_path = str(data_dir)
fc.filter_pattern = '*.czi'
display(fc)
# Select a CZI file and confirm it before running the next cell.

FileChooser(path='F:\Pixi_Projects\zen_czi\smartmicroscopy_2026-11\data', filename='', title='', show_hidden=F…

In [ ]:
# get only specific metadata
if fc.selected is None:
    raise ValueError("Select a CZI file in the file chooser and confirm it before running this cell.")
filepath = str(fc.selected)

czi_dimensions = czimd.CziDimensions(filepath)
print("SizeS: ", czi_dimensions.SizeS)
print("SizeT: ", czi_dimensions.SizeT)
print("SizeZ: ", czi_dimensions.SizeZ)
print("SizeC: ", czi_dimensions.SizeC)
print("SizeY: ", czi_dimensions.SizeY)
print("SizeX: ", czi_dimensions.SizeX)

In [ ]:
# and get more info
czi_scaling = czimd.CziScaling(filepath)
czi_channels = czimd.CziChannelInfo(filepath)
czi_bbox = czimd.CziBoundingBox(filepath)
czi_objectives = czimd.CziObjectives(filepath)
czi_detectors = czimd.CziDetector(filepath)
czi_microscope = czimd.CziMicroscope(filepath)
czi_sample = czimd.CziSampleInfo(filepath)

# show all attributes of the microscope (adjust info_to_show to another object if needed)
info_to_show =  czi_microscope

for attr in dir(info_to_show):
    if not attr.startswith("__"):
        print(f"{attr}: {getattr(info_to_show, attr)}")

In [ ]:
# get the complete metadata at once as one big class
mdata = czimd.CziMetadata(filepath)

# convert metadata dictionary to a pandas dataframe
mdframe = misc.md2dataframe(mdata, reduced_params=True)

# create a ipywdiget to show the dataframe with the metadata
wd1 = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd1:
    display(HTML(mdframe.to_html()))

display(widgets.VBox(children=[wd1]))

In [ ]:
# write XML to disk
xmlfile = czimd.writexml(filepath)
print("XML File written to:", xmlfile)

In [ ]:
# get the planetable for the CZI file
pt, savepath = planetable.get_planetable(filepath,
                                         norm_time=True,
                                         save_table=True,
                                         planes={"time": 0, "channel": 0}
                                        )

wd2 = widgets.HTML(
    value=pt.to_html(),
    layout=widgets.Layout(height="300px", overflow="auto"),
)
display(wd2)